In [2]:
!pip install beautifulsoup4


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import requests

url = "https://www.gov.uk/search/news-and-communications"
page = requests.get(url)
html = page.content
# Voir le code html source
print(page.content)

b'\n\n<!DOCTYPE html>\n<html class="govuk-template govuk-template--rebranded" lang="en">\n  <head>\n    <meta charset="utf-8">\n    <title>News and communications - GOV.UK</title>\n\n    <script src="/assets/finder-frontend/govuk_publishing_components/vendor/lux/lux-measurer-d66192b1b665d415e1874e49a48138b7ae339548aa6cdb8687c7f9933e72b969.js" async="async"></script>\n    <script src="/assets/finder-frontend/govuk_publishing_components/rum-custom-data-b9e1806d1da2fa8ef1855d01aa71ba5eb0afc010dd2c4de3672cc7c7a39b0c8c.js" type="module"></script>\n    <script src="/assets/finder-frontend/govuk_publishing_components/rum-loader-a65b10e18ceeba3bd8a2eac507c7f2c513cdc82f35097df903fdea87f1dc2e33.js" async="async" data-lux-reporter-script="/assets/finder-frontend/govuk_publishing_components/vendor/lux/lux-reporter-16e474d53ab9842457c92448e9ea0e6faea553d7b8250f940eec8c50285b48af.js"></script>\n\n    <meta name="govuk:components_gem_version" content="66.4.1">\n    <script src="/assets/finder-fronten

In [4]:
import requests
from bs4 import BeautifulSoup
from bs4 import BeautifulSoup

with open("index.html", "r") as file:
   soup = BeautifulSoup(file.read(), 'html.parser')


In [5]:
# afficher le titre de la page :
print(soup.title.text)

Exercice extraction HTML


In [6]:
print(soup.find_all('h2'))

[<h2>Produit 1</h2>, <h2>Produit 2</h2>, <h2>Produit 3</h2>]


In [7]:
title = soup.title
print(title.string)

Exercice extraction HTML


In [8]:
title_text = str(soup.find_all('h1'))
print(title_text)

[<h1 id="titre">Bienvenue sur notre site web</h1>]


In [9]:
liste_produits = soup.find_all("li")
print(liste_produits)

[<li class="product">
<h2>Produit 1</h2>
<p class="price">Prix: 10€</p>
<p>Description : Lorem ipsum dolor sit amet, consectetur adipiscing elit.</p>
</li>, <li class="product">
<h2>Produit 2</h2>
<p class="price">Prix: 20€</p>
<p>Description : Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua.</p>
</li>, <li class="product">
<h2>Produit 3</h2>
<p class="price">Prix: 30€</p>
<p>Description : Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat.</p>
</li>]


In [10]:
noms_produits = soup.find_all('h2')
print(noms_produits)

[<h2>Produit 1</h2>, <h2>Produit 2</h2>, <h2>Produit 3</h2>]


In [11]:
prix_produits = soup.find_all(class_="price")

# for balise in prix_produits:
#     prix = balise.text.strip()
#     print(prix)
    
print(prix_produits)

[<p class="price">Prix: 10€</p>, <p class="price">Prix: 20€</p>, <p class="price">Prix: 30€</p>]


In [12]:
import re
descriptions = soup.find_all("p", string=re.compile("Description"))
#print(descriptions)
descriptions_dico = {}
compteur = 1


for desc in descriptions:
    texte = desc.text.strip()
    if "Description" in texte:
        # On crée une clé unique, ex: "Produit_1"
        cle = f"Description_{compteur}" 
        descriptions_dico[cle] = texte
        compteur += 1

In [13]:
print(descriptions_dico)

{'Description_1': 'Description : Lorem ipsum dolor sit amet, consectetur adipiscing elit.', 'Description_2': 'Description : Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua.', 'Description_3': 'Description : Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat.'}


1. **Extraction** : réutiliser le code précédent pour extraire les informations des produits (nom, prix, description) et créer un **dictionnaire** associé.
2. **Affichage** des données extraites.
3. **Transformation** : convertir les prix des euros vers les dollars selon la formule `dollar = euro * 1.2`, puis ajouter cette information au dictionnaire de chaque produit.
4. **Affichage** des données transformées.

In [14]:
# 1) EXTRACTION : on parcourt chaque <li class="product"> et on construit un dictionnaire par produit
import re

produits = {}

for i, produit in enumerate(soup.find_all("li", class_="product"), start=1):
    nom = produit.find("h2").text.strip()

    # Le prix est dans <p class="price">Prix: 10€</p> -> on récupère le nombre
    texte_prix = produit.find("p", class_="price").text
    prix_euro = float(re.search(r"(\d+(?:[.,]\d+)?)", texte_prix).group(1).replace(",", "."))

    # La description est le <p> contenant le mot "Description"
    description = produit.find("p", string=re.compile("Description")).text.strip()

    produits[f"Produit_{i}"] = {
        "nom": nom,
        "prix_euro": prix_euro,
        "description": description,
    }

produits


{'Produit_1': {'nom': 'Produit 1',
  'prix_euro': 10.0,
  'description': 'Description : Lorem ipsum dolor sit amet, consectetur adipiscing elit.'},
 'Produit_2': {'nom': 'Produit 2',
  'prix_euro': 20.0,
  'description': 'Description : Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua.'},
 'Produit_3': {'nom': 'Produit 3',
  'prix_euro': 30.0,
  'description': 'Description : Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat.'}}

In [15]:
# 2) AFFICHAGE des données extraites
print("=== Données extraites ===")
for cle, infos in produits.items():
    print(f"{cle} : {infos['nom']} | {infos['prix_euro']} € | {infos['description']}")


=== Données extraites ===
Produit_1 : Produit 1 | 10.0 € | Description : Lorem ipsum dolor sit amet, consectetur adipiscing elit.
Produit_2 : Produit 2 | 20.0 € | Description : Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua.
Produit_3 : Produit 3 | 30.0 € | Description : Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat.


In [16]:
# 3) TRANSFORMATION : conversion euros -> dollars (dollar = euro * 1.2)
TAUX = 1.2

for infos in produits.values():
    infos["prix_dollar"] = round(infos["prix_euro"] * TAUX, 2)

produits


{'Produit_1': {'nom': 'Produit 1',
  'prix_euro': 10.0,
  'description': 'Description : Lorem ipsum dolor sit amet, consectetur adipiscing elit.',
  'prix_dollar': 12.0},
 'Produit_2': {'nom': 'Produit 2',
  'prix_euro': 20.0,
  'description': 'Description : Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua.',
  'prix_dollar': 24.0},
 'Produit_3': {'nom': 'Produit 3',
  'prix_euro': 30.0,
  'description': 'Description : Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat.',
  'prix_dollar': 36.0}}

In [17]:
# 4) AFFICHAGE des données transformées
print("=== Données transformées (euros -> dollars) ===")
for cle, infos in produits.items():
    print(f"{cle} : {infos['nom']} | {infos['prix_euro']} € = {infos['prix_dollar']} $")


=== Données transformées (euros -> dollars) ===
Produit_1 : Produit 1 | 10.0 € = 12.0 $
Produit_2 : Produit 2 | 20.0 € = 24.0 $
Produit_3 : Produit 3 | 30.0 € = 36.0 $
